In [2]:
import os
import fitz  # PyMuPDF
from tqdm import tqdm
DATA_PATH = "./data/"

def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    text = ""
    for page in doc:
        text += page.get_text()
    return text

    
documents = []
for file in tqdm(os.listdir(DATA_PATH)):
    if file.endswith(".pdf"):
        full_path = os.path.join(DATA_PATH, file)
        text = extract_text_from_pdf(full_path)
        documents.append({
            "filename": file,
            "text": text
        })

print("Total documents loaded:", len(documents))

100%|████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:04<00:00,  1.66s/it]

Total documents loaded: 3


## STRATEGY 1 — Fixed Chunking (Baseline)

In [4]:
def chunk_fixed(text, chunk_size=500):
    return [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]

## STRATEGY 2 — Sliding Window Chunking

In [6]:
def chunk_sliding(text, chunk_size=500, overlap=100):
    chunks = []
    start = 0
    
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start = end - overlap
        
    return chunks

## STRATEGY 3 — Recursive Chunking (Smart Split)

In [9]:
import re

def recursive_chunk(text, max_length=500):
    
    paragraphs = text.split("\n\n")
    chunks = []
    
    for para in paragraphs:
        if len(para) <= max_length:
            chunks.append(para)
        else:
            sentences = re.split(r'(?<=[.!?]) +', para)
            current_chunk = ""
            
            for sentence in sentences:
                if len(current_chunk) + len(sentence) <= max_length:
                    current_chunk += " " + sentence
                else:
                    chunks.append(current_chunk.strip())
                    current_chunk = sentence
            
            if current_chunk:
                chunks.append(current_chunk.strip())
    
    return chunks

## STRATEGY 4 — Semantic Chunking (Advanced)

In [10]:
from sentence_transformers import util

def semantic_chunk(text, model, threshold=0.7):
    
    sentences = re.split(r'(?<=[.!?]) +', text)
    sentence_embeddings = model.encode(sentences)
    
    chunks = []
    current_chunk = sentences[0]
    
    for i in range(1, len(sentences)):
        
        sim = util.cos_sim(sentence_embeddings[i-1], sentence_embeddings[i])
        
        if sim > threshold:
            current_chunk += " " + sentences[i]
        else:
            chunks.append(current_chunk)
            current_chunk = sentences[i]
    
    chunks.append(current_chunk)
    
    return chunks

## COMPARISON PIPELINE

In [12]:
def build_chunks(strategy="fixed"):
    
    all_chunks = []
    
    for doc in documents:
        
        if strategy == "fixed":
            chunks = chunk_fixed(doc["text"])
            
        elif strategy == "sliding":
            chunks = chunk_sliding(doc["text"])
            
        elif strategy == "recursive":
            chunks = recursive_chunk(doc["text"])
            
        elif strategy == "semantic":
            chunks = semantic_chunk(doc["text"], model)
            
        for chunk in chunks:
            all_chunks.append({
                "filename": doc["filename"],
                "content": chunk
            })
    
    return all_chunks

## Evaluate Retrieval

In [18]:
from sentence_transformers import SentenceTransformer
import numpy as np
import faiss

model = SentenceTransformer("all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [19]:
def evaluate_query(query, strategy):
    
    chunks = build_chunks(strategy)
    
    texts = [c["content"] for c in chunks]
    embeddings = model.encode(texts)
    
    index = faiss.IndexFlatL2(embeddings.shape[1])
    index.add(np.array(embeddings))
    
    query_embedding = model.encode([query])
    
    distances, indices = index.search(np.array(query_embedding), 3)
    
    results = [chunks[i]["content"][:200] for i in indices[0]]
    
    return results

In [20]:
query = "Explain salary details in LTIMindtree"

print("FIXED:\n", evaluate_query(query, "fixed"))
print("SLIDING:\n", evaluate_query(query, "sliding"))
print("RECURSIVE:\n", evaluate_query(query, "recursive"))
print("SEMANTIC:\n", evaluate_query(query, "semantic"))

FIXED:
 ['umbai - 400 001. INDIA\n          www.ltimindtree.com    |   E-mail:info@ltimindtree.com   |   CIN:L72900MH1996PLC104693\nLTIMindtree Limited is a subsidiary of Larsen & Toubro Limited\n©LTIMindtree|Conf', 'and the payment of \nallowances will be governed by the rules and regulations of the Company as may be applicable from time to time.\nPage 6 of 7\n          LTIMindtree Limited \n          Corporate Offic', 'ndtree.com   |   CIN:L72900MH1996PLC104693\nLTIMindtree Limited is a subsidiary of Larsen & Toubro Limited\n©LTIMindtree|Confidential 2025\n                                         ANNEXURE-1\n Eligibilit']
SLIDING:
 ['2900MH1996PLC104693\nLTIMindtree Limited is a subsidiary of Larsen & Toubro Limited\n©LTIMindtree|Confidential 2025\n                                                       ANNEXURE-4\n                    ', '3\nLTIMindtree Limited is a subsidiary of Larsen & Toubro Limited\n©LTIMindtree|Confidential 2025\n8.\nYou may be confirmed in 3 months from the eff